# Traffic-Law QA — Text retrieval, OCR and generation exploration

**Trương Minh Hoàng · MSSV 25410008**

**ARCHIVED_SOURCE — source review, not a fresh benchmark.** Bản học tập lịch sử được giữ để giải thích quá trình làm bài. Output cũ và thông tin môi trường riêng đã được bỏ; xem README/AUDIT của dự án trước khi chạy. Historical calculations and known limitations are documented separately.

 # 1. Tư Duy Giải Quyết Bài Toán


Mục tiêu chính: Xây dựng hệ thống hỏi đáp đa phương thức (văn bản + hình ảnh) về luật giao thông Việt Nam, với:

*   Đầu vào: Câu hỏi dạng text hoặc ảnh biển báo.

*   Đầu ra: Câu trả lời chính xác + điều luật tham chiếu + độ tin cậy.


**# Tư duy tiếp cận:**

*   Không dùng LLM lớn → Ưu tiên model nhẹ + kỹ thuật tối ưu.

*   Kết hợp Retrieval-Augmented Generation (RAG): Dùng cơ sở dữ liệu luật để bổ sung context, giảm hallucination.

*   Đa phương thức: Xử lý cả text và ảnh, nhưng ưu tiên text-first (do ảnh biển báo có thể chuyển đổi thành text qua OCR).

# 2. Các Bước Triển Khai Chi Tiết

Bước 1: Chuẩn Bị Dữ Liệu
Yêu cầu:

Dataset từ BTC VISP 2025 (đã có sẵn file visp2025_law.json và ảnh biển báo).

Cần làm sạch và tổ chức lại dữ liệu.

Công việc:

Tiền xử lý văn bản luật

In [ ]:
import os

# Define the path to the dataset folder
dataset_folder_path = 'data/local'

# Check if the directory exists
if os.path.exists(dataset_folder_path):
    print(f"Contents of the directory: {dataset_folder_path}")
    # List all files and directories in the specified path
    contents = os.listdir(dataset_folder_path)
    for item in contents:
        item_path = os.path.join(dataset_folder_path, item)
        if os.path.isdir(item_path):
            print(f"Directory: {item}")
        else:
            print(f"File: {item}")
else:
    print(f"Error: The directory '{dataset_folder_path}' was not found. Please check the path.")

In [ ]:
!pip install torch transformers sentence-transformers faiss-cpu pytesseract pillow pandas numpy scikit-learn gradio

In [ ]:
import json
import pandas as pd

# Assuming the file was uploaded as 'vlsp2025_law.json'
file_name = 'data/local/law_db/vlsp2025_law_new.json'

try:
    with open(file_name, 'r', encoding='utf-8') as f:
        law_data = json.load(f)

    # Convert the JSON data to a pandas DataFrame for easier handling
    law_df = pd.DataFrame(law_data)

    print(f"Successfully loaded {file_name} into a pandas DataFrame.")
    display(law_df.head())

except FileNotFoundError:
    print(f"Error: The file '{file_name}' was not found. Please make sure it's uploaded correctly.")
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from '{file_name}'. Please check the file format.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import numpy as np

# Extract the 'law_text' column
law_texts = processed_law_df['law_text']

# Generate embeddings for the law texts
# The encode method handles batching automatically
law_embeddings = text_encoder_model.encode(law_texts.tolist(), show_progress_bar=True)

# Store the embeddings as a NumPy array (already is a NumPy array by default)
print(f"Generated embeddings with shape: {law_embeddings.shape}")

# Display the first few embeddings as an example
print("\nFirst 5 embeddings:")
print(law_embeddings[:5])

code Colab cho T4 16 GB để sinh output dài 300–500 token, vẫn ổn định mà không crash, dùng offload + quantization

In [ ]:
# Kiểm tra cấu trúc dữ liệu
print(law_df.columns)

# The 'articles' column contains a list of dictionaries.
# We need to extract the article information from these dictionaries.
# Let's create a new list of dictionaries with flattened information.
processed_articles = []
for index, row in law_df.iterrows():
    law_id = row['id']
    law_title = row['title']
    articles_list = row['articles']
    for article in articles_list:
        # Assuming each article dictionary has 'id' and 'text' keys
        processed_articles.append({
            'law_id': law_id,
            'law_title': law_title,
            'article_id': article.get('id'), # Use .get() for safer access
            'article_text': article.get('text') # Use .get() for safer access
        })

# Create a new DataFrame from the processed articles
processed_law_df = pd.DataFrame(processed_articles)

# Create the 'law_text' column
processed_law_df['law_text'] = "Điều " + processed_law_df['article_id'].astype(str) + ": " + processed_law_df['article_text']


# Display the first few rows of the processed DataFrame
display(processed_law_df.head())

# Lưu dữ liệu đã xử lý (optional, if you want to save this flattened data)
# processed_law_df.to_csv('processed_law.csv', index=False)

In [ ]:
# Optional model authentication is configured outside this historical notebook.

In [ ]:
# --- Cài đặt thư viện cần thiết ---
!pip install huggingface_hub
!pip install -q transformers sentence-transformers faiss-cpu torch sentencepiece accelerate bitsandbytes
!pip install -U transformers

# --- Import ---
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from huggingface_hub import login

# --- Hugging Face login (nếu model private / cần accept terms) ---
login(new_session=False)  # nhập token Hugging Face

# --- 1️⃣ Load PhoGPT-4B-Chat ---
try:
    print("Attempting to load vinai/PhoGPT-4B model and tokenizer...")

    # Cấu hình quantization 8-bit + offload sang CPU cho các module cần FP32
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )

    tokenizer = AutoTokenizer.from_pretrained("vinai/PhoGPT-4B", cache_dir="./model_cache")

    model = AutoModelForCausalLM.from_pretrained(
        "vinai/PhoGPT-4B",
        quantization_config=bnb_config,
        device_map="auto",         # phân bổ tự động GPU/CPU
        offload_folder="offload",  # lưu module offload sang RAM/CPU
        cache_dir="./model_cache"
    )

    print("Successfully loaded vinai/PhoGPT-4B model and tokenizer.")

except Exception as e:
    print(f"Error loading vinai/PhoGPT-4B model: {e}")
    print("Ensure you have access on Hugging Face and are authenticated.")
    tokenizer = None
    model = None

# --- 2️⃣ Load Sentence Transformer encoder (Vietnamese SBERT) ---
try:
    print("\nAttempting to load Sentence Transformer encoder: keepitreal/vietnamese-sbert...")
    encoder = SentenceTransformer('keepitreal/vietnamese-sbert')
    print("Successfully loaded Sentence Transformer encoder.")

except Exception as e:
    print(f"Error loading Sentence Transformer encoder: {e}")
    encoder = None

Load PhoBERT làm Text Encoder (Retrieval)



In [ ]:
from sentence_transformers import SentenceTransformer

# Sử dụng model SBERT tiếng Việt (dựa trên PhoBERT)
encoder = SentenceTransformer('keepitreal/vietnamese-sbert')  # Hoặc 'vinai/phobert-base'

# Ví dụ mã hóa văn bản
law_texts = ["Điều 15: Cấm xe tải trên 2.5 tấn", "Điều 20: Xe máy được phép lưu thông"]
law_embeddings = encoder.encode(law_texts)

3. Xây dựng FAISS Index

In [ ]:
import faiss
import numpy as np

# Tạo index
index = faiss.IndexFlatIP(768)  # 768 là chiều vector của PhoBERT
index.add(np.array(law_embeddings).astype('float32'))

# Hàm truy vấn
def retrieve_laws(question, top_k=3):
    question_embedding = encoder.encode([question])
    distances, indices = index.search(question_embedding, top_k)
    return indices[0]

4. Load PhoBERT làm Generator

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Hàm generate text theo chunk ---
def generate_answer(question, context, max_total_tokens=400, chunk_size=100):
    prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    all_generated_ids = input_ids.clone()  # lưu toàn bộ token sinh ra
    total_generated = 0

    while total_generated < max_total_tokens:
        outputs = model.generate(
            input_ids,
            max_new_tokens=chunk_size,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
        # Lấy phần sinh ra mới
        new_tokens = outputs[0, input_ids.shape[-1]:]
        if new_tokens.shape[0] == 0:
            break  # hết token sinh ra
        all_generated_ids = torch.cat([all_generated_ids, new_tokens], dim=-1)
        input_ids = new_tokens.unsqueeze(0)  # input cho chunk tiếp theo
        total_generated += new_tokens.shape[0]

    return tokenizer.decode(all_generated_ids[0], skip_special_tokens=True)

5. Ghép thành RAG Pipeline

In [ ]:
def rag_pipeline(question):
    # Bước 1: Truy xuất luật
    law_indices = retrieve_laws(question)
    context = "\n".join([law_texts[i] for i in law_indices])

    # Bước 2: Sinh câu trả lời
    answer = generate_answer(question, context)

    return {
        "answer": answer,
        "related_laws": [law_texts[i] for i in law_indices]
    }

# Demo
question = "Xe máy có được đi vào đường cấm xe tải không?"
result = rag_pipeline(question)
print("Câu trả lời:", result["answer"])
print("Luật liên quan:", result["related_laws"])

6. Tối ưu hóa cho Colab Free

In [ ]:
#a. Sử dụng GPU miễn phí
import torch
print("GPU available:", torch.cuda.is_available())

In [ ]:
#b. Giảm bộ nhớ
# Dùng mixed precision
model = model.half()  # Chuyển sang float16

In [ ]:
#c. Cache model
from google.colab import drive
drive.mount('/content/drive')

# Lưu model vào Google Drive
model.save_pretrained("/content/drive/MyDrive/phobert_rag")
tokenizer.save_pretrained("/content/drive/MyDrive/phobert_rag")

7. Demo với dữ liệu thực tế

In [ ]:
# Load dataset luật từ file JSON (đã upload lên Colab)
import json
with open("luat_giao_thong.json", "r", encoding="utf-8") as f:
    laws = json.load(f)

law_texts = [f"Điều {item['article_number']}: {item['content']}" for item in laws]
law_embeddings = encoder.encode(law_texts)
index.add(np.array(law_embeddings).astype('float32'))

# Hỏi đáp
question = "Tốc độ tối đa trong khu đô thị là bao nhiêu?"
print(rag_pipeline(question))

8. Xử lý ảnh biển báo (nếu cần)

In [ ]:
!pip install -q pytesseract opencv-python
import pytesseract
from PIL import Image

def image_to_text(image_path):
    text = pytesseract.image_to_string(Image.open(image_path), lang='vie')
    return text

# Tích hợp vào RAG
def process_image_question(image_path, question=""):
    text = image_to_text(image_path)
    return rag_pipeline(f"{question} (Nội dung ảnh: {text})")

Bước 4: Kết Hợp Retriever + Generator (RAG Pipeline)

In [ ]:
# Placeholder for the retrieve_laws function
# This function will be implemented in a later step using the Text Encoder and FAISS index
def retrieve_laws(question):
    """
    Retrieves relevant law articles based on the user's question.

    Args:
        question (str): The user's question (can be text extracted from image).

    Returns:
        list: A list of strings, where each string is a relevant law article.
    """
    # For now, returning the placeholder laws used in the previous cell
    # Replace this with actual retrieval logic later
    print("Note: Using placeholder laws for retrieval. Implement actual retrieval logic.")
    return [
        "Điều 10: Các hành vi bị nghiêm cấm khi tham gia giao thông đường bộ.",
        "Điều 11: Quy tắc chung về báo hiệu đường bộ.",
        "Điều 12: Tốc độ xe cơ giới, xe máy chuyên dùng.",
        "Điều 13: Sử dụng làn đường.",
        "Điều 14: Vượt xe.",
        "Điều 15: Chuyển hướng xe.",
        "Điều 16: Lùi xe.",
        "Điều 17: Tránh xe đi ngược chiều.",
        "Điều 18: Dừng xe, đỗ xe trên đường bộ.",
        "Điều 19: Quay đầu xe.",
    ]


def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        # Input is likely an image file path, perform OCR
        print(f"Image input detected: {input_data}. Performing OCR...")
        # Ensure extract_text_from_image is defined (from cell CsNFgNn5sBIJ)
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        # Input is a text question
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question (extracted text or original text)
    related_laws = retrieve_laws(question)
    context = "\n".join(related_laws)

    # Step 3: Generate answer using the question and context
    # Ensure the generate_answer function is defined (from cell MrgWbpEjmvJs)
    if 'generate_answer' not in globals():
        return {"error": "generate_answer function not defined. Please run the cell that defines it."}
    answer = generate_answer(question, context)

    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": 0.9  # Có thể tính từ similarity score của FAISS
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
# Check if model and tokenizer are loaded before running the pipeline
if 'tokenizer' in globals() and tokenizer is not None and 'model' in globals() and model is not None:
    print("--- Testing with text input ---")
    result_text = rag_pipeline(question_text)
    print("Kết quả (Text Input):", result_text)
else:
    print("Skipping text input demo. Language model not loaded.")


# Demo with image input (using an example image from your Google Drive)
# Ensure you have run the Google Drive mounting and listing cells first
# and that law_image_files list is available and not empty
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0] # Use the first image from the law_db folder
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing with image input: {example_image_path} ---")
    # Check if necessary functions are defined before attempting image processing
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")

Bước 5: Đánh Giá Hiệu Năng

In [ ]:
from sklearn.metrics import accuracy_score

# Giả sử có test_data gồm các cặp (câu hỏi, đáp án đúng)
# Replace with your actual test data later
test_questions = ["Câu hỏi 1", "Câu hỏi 2"]
test_answers = ["A", "B"]  # Đáp án đúng (example)

predicted_answers = []
# Ensure rag_pipeline is defined and functional before running this loop
if 'rag_pipeline' in globals():
    for q in test_questions:
        # Call the RAG pipeline to get the predicted answer
        # The rag_pipeline currently uses placeholder retrieval, so results will not be accurate yet.
        result = rag_pipeline(q)
        # Extract the predicted answer from the result.
        # You might need to adjust this based on the actual output format of your generate_answer function.
        # For now, assuming the answer is the first character as in the original code's logic.
        # This part likely needs adjustment based on how your generator provides the answer.
        if result and "answer" in result and result["answer"]:
             # This assumes the answer is just the generated text.
             # For actual evaluation, you might need a way to compare the generated text
             # with the expected answer, possibly involving some form of semantic comparison or keyword matching.
             # For this placeholder, we will just append the full generated answer text.
             # A more robust evaluation metric (like ROUGE, BLEU, or a custom metric) might be needed.
             predicted_answers.append(result["answer"])
        else:
             predicted_answers.append("") # Append empty string if no answer is generated


    # The accuracy_score metric as used here (comparing single characters 'A'/'B')
    # is likely not suitable for evaluating free-form text answers generated by an LLM.
    # You will need to define appropriate evaluation metrics and comparison logic
    # that match the nature of the generated answers and expected correct answers.

    # For demonstration, let's print the predicted answers
    print("Test Questions:", test_questions)
    print("Predicted Answers:", predicted_answers)
    # print("Correct Answers:", test_answers) # Uncomment to see correct answers if you have them

    # Note: Calculating a simple accuracy_score like this is likely incorrect for text generation evaluation.
    # More advanced metrics are usually required.
    # The following line is kept from the original code but is likely not appropriate
    # for evaluating free-form text generation without a proper comparison method.
    # accuracy = accuracy_score(test_answers, predicted_answers) # This will likely cause an error or be meaningless
    # print(f"Accuracy: {accuracy * 100:.2f}%")

    print("\nNote: The accuracy score calculated here (if uncommented) will not be meaningful")
    print("until you implement the actual retrieval and have a test set designed for comparing generated text.")
    print("Consider using metrics like ROUGE or BLEU for evaluating text generation.")

else:
    print("Skipping performance evaluation because rag_pipeline is not defined or not functional.")

# Tích Hợp Xử Lý Ảnh Biển Báo Bằng OpenCV + Tesseract OCR

In [ ]:
!pip install opencv-python pytesseract pillow
!sudo apt install tesseract-ocr  # Trên Colab/Kaggle (Linux)
!sudo apt install libtesseract-dev  # Nếu cần

**Tiền Xử Lý Ảnh Với OpenCV
Mục tiêu: Cải thiện chất lượng ảnh trước khi đưa vào OCR.**

In [ ]:
import cv2
import numpy as np
from PIL import Image

def preprocess_image(image_path):
    # Đọc ảnh bằng OpenCV
    img = cv2.imread(image_path)

    # Check if image was loaded successfully
    if img is None:
        print(f"Error: Image not loaded from {image_path}. Please check the file path.")
        return None # Return None or raise an error if image loading fails

    # Chuyển sang ảnh xám
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Làm nét ảnh (adaptive thresholding)
    thresh = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )

    # Khử nhiễu (morphological opening)
    kernel = np.ones((2, 2), np.uint8)
    processed = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    return processed

# Ví dụ
# Make sure 'bien_bao.jpg' exists in the correct directory or provide the full path
processed_img = preprocess_image("bien_bao.jpg")

if processed_img is not None:
    Image.fromarray(processed_img).save("processed_bien_bao.jpg")  # Lưu ảnh đã xử lý
    print("Processed image saved as processed_bien_bao.jpg")
else:
    print("Image processing skipped due to loading error.")

Bước 2: Chọn Model & Kỹ Thuật Tối Ưu
Lựa chọn model (đảm bảo open-source, nhỏ gọn): *(Model trên HuggingFace)*
*   **Text Encoder**	*keepitreal/vietnamese-sbert*	: Mã hóa câu hỏi và luật để tìm
*   **LLM Generate**	*vinai/phobert-base (hoặc tinyllama-1.1B)*	Trả lời câu hỏi dựa trên context
*   **Image Encoder**	*dangvantuan/vit-base-patch16-224-vietnamese*	Nhận diện biển báo từ ảnh
Kỹ thuật tối ưu:

**1. Retrieval-Augmented Generation (RAG):**
     
     Dùng FAISS để tìm luật liên quan nhanh.

**2. Quantization (Giảm kích thước model):**
      
      Dùng bitsandbytes để nén model xuống 4-bit

# Bước 3: Xây Dựng Pipeline Xử Lý
Pipeline tổng thể:

1.   Nhận input (text hoặc ảnh).
2.   Nếu là ảnh: Dùng OCR hoặc Vision Transformer để chuyển thành text.
3.   Tìm luật liên quan bằng FAISS/SBERT.
4.   Generate câu trả lời bằng LLM nhỏ (kèm độ tin cậy).

# Bước 4: Đánh Giá & Tối Ưu
Metrics cần đo:

Độ chính xác (Accuracy): Tỉ lệ câu trả lời đúng.

Thời gian inference: Đảm bảo <1s/câu hỏi.

Độ tin cậy (Confidence Score): Kiểm tra xem model có "tự tin" khi trả lời không.

**Tối ưu hóa:**

*   Fine-tuning trên dataset luật giao thông cụ thể.
*   Pruning: Loại bỏ các layer không cần thiết của model.
*   Cache kết quả truy vấn luật để tăng tốc độ.

# Bước 5: Triển Khai Demo
Lựa chọn deploy:

1.   Trên Colab
2.   Lên Hugging Face Spaces: Đóng gói thành repository và deploy




In [ ]:
!pip install torch transformers sentence-transformers faiss-cpu pytesseract pillow pandas numpy scikit-learn gradio

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to the train_images folder
train_images_path = 'data/local/train_data/train_images'

# Check if the directory exists
if os.path.exists(train_images_path):
    print(f"Accessing image folder: {train_images_path}")
    # List the files in the folder
    image_files = [f for f in os.listdir(train_images_path) if os.path.isfile(os.path.join(train_images_path, f))]
    print(f"Found {len(image_files)} image files.")
    # Display the first few file names
    print("First 5 image files:", image_files[:5])

    # Example of how to load one image using your preprocess_image function
    if image_files:
        example_image_path = os.path.join(train_images_path, image_files[0])
        print(f"\nLoading and preprocessing example image: {example_image_path}")
        # Assuming preprocess_image is defined in a previous cell
        if 'preprocess_image' in globals():
            processed_example_img = preprocess_image(example_image_path)

            if processed_example_img is not None:
                print("Example image loaded and preprocessed successfully.")
                # You can now work with processed_example_img
                # For example, you could display it or pass it to your OCR function
                # from PIL import Image
                # display(Image.fromarray(processed_example_img)) # Requires matplotlib or similar for display in Colab
            else:
                 print("Failed to preprocess the example image.")
        else:
            print("Error: preprocess_image function not found. Please run the cell that defines it.")

    else:
        print("No image files found in the specified directory.")

else:
    print(f"Error: The directory '{train_images_path}' was not found. Please check the path.")

# 3. Trích Xuất Chữ Bằng Tesseract OCR

In [ ]:
import pytesseract
from PIL import Image
import os # Import os to handle file paths

def extract_text_from_image(image_path, lang='vie'):
    # Tiền xử lý ảnh
    processed_img = preprocess_image(image_path)

    # Check if image preprocessing was successful
    if processed_img is None:
        print(f"Error: Preprocessing failed for image {image_path}. Cannot perform OCR.")
        return "" # Return an empty string or handle as appropriate

    # Convert OpenCV image (NumPy array) to PIL Image
    # Tesseract works reliably with PIL Images
    processed_img_pil = Image.fromarray(processed_img)


    # Dùng Tesseract OCR để trích xuất chữ
    # Ensure pytesseract executable is in your PATH if not using default installation
    # In Colab, the apt install command usually handles this.
    try:
        text = pytesseract.image_to_string(
            processed_img_pil, # Use the PIL image
            lang=lang,  # Ngôn ngữ Tiếng Việt
            config='--psm 6'  # Chế độ phân tích đoạn văn
        )
        return text.strip()
    except pytesseract.TesseractNotFoundError:
        print("Error: Tesseract is not installed or not in your PATH.")
        print("Please ensure you have run: !sudo apt install tesseract-ocr")
        return ""
    except Exception as e:
        print(f"An error occurred during OCR processing: {e}")
        return ""


# Ví dụ
# Use an example image path from your Google Drive folder
# Make sure you have run the cell to mount Google Drive and list files first
train_images_path = 'data/local/train_data/train_images'
# Assuming image_files list was created in the previous cell
if 'image_files' in globals() and image_files:
    example_image_name = image_files[0] # Use the first image found
    example_image_path = os.path.join(train_images_path, example_image_name)
    print(f"Attempting to extract text from: {example_image_path}")
    text_from_image = extract_text_from_image(example_image_path)
    print("Text trích xuất:", text_from_image)
else:
    print("Error: No image files found or image_files list is not available.")
    print("Please run the Google Drive mounting and listing cell first.")

In [ ]:
from google.colab import drive
import os
from PIL import Image # Import Image for displaying

# Mount Google Drive (if not already mounted)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

# Define the path to the new image folder from the law database
law_images_path = 'data/local/law_db/images.fld'

# Check if the directory exists
if os.path.exists(law_images_path):
    print(f"Accessing image folder: {law_images_path}")
    # List the files in the folder
    law_image_files = [f for f in os.listdir(law_images_path) if os.path.isfile(os.path.join(law_images_path, f))]
    print(f"Found {len(law_image_files)} image files.")
    # Display the first few file names
    print("First 5 law image files:", law_image_files[:5])

    # Example of how to load one image using your preprocess_image function
    if law_image_files:
        example_law_image_name = law_image_files[0] # Use the first image found
        example_law_image_path = os.path.join(law_images_path, example_law_image_name)
        print(f"\nLoading and preprocessing example law image: {example_law_image_path}")
        # Assuming preprocess_image is defined in a previous cell
        if 'preprocess_image' in globals():
            processed_example_law_img = preprocess_image(example_law_image_path)

            if processed_example_law_img is not None:
                print("Example law image loaded and preprocessed successfully.")
                # Display the processed image (requires matplotlib or similar)
                # If you want to display the image, you'll need to import matplotlib.pyplot as plt
                # and use plt.imshow() or display(Image.fromarray(processed_example_law_img))
                print("You can display this image using PIL and display():")
                display(Image.fromarray(processed_example_law_img))

            else:
                 print("Failed to preprocess the example law image.")
        else:
            print("Error: preprocess_image function not found. Please run the cell that defines it.")

    else:
        print("No image files found in the specified law database directory.")

else:
    print(f"Error: The directory '{law_images_path}' was not found. Please check the path.")

# Task
Implement a Retriever component for the RAG pipeline using the "keepitreal/vietnamese-sbert" model and FAISS, and integrate it into the existing RAG pipeline.

## Load text encoder model

### Subtask:
Load the `keepitreal/vietnamese-sbert` model using the `sentence-transformers` library.


**Reasoning**:
The subtask is to load the text encoder model. This requires importing the necessary class and then loading the model.



In [ ]:
from sentence_transformers import SentenceTransformer

# Load the text encoder model
text_encoder_model = SentenceTransformer("keepitreal/vietnamese-sbert")

print("Successfully loaded the text encoder model: keepitreal/vietnamese-sbert")

## Generate embeddings

### Subtask:
Generate vector embeddings for each law article in the `processed_law_df` DataFrame using the loaded Text Encoder model.


**Reasoning**:
Generate vector embeddings for each law article using the loaded text encoder model and store them as a NumPy array.



In [ ]:
import numpy as np

# Extract the 'law_text' column
law_texts = processed_law_df['law_text']

# Generate embeddings for the law texts
# The encode method handles batching automatically
law_embeddings = text_encoder_model.encode(law_texts.tolist(), show_progress_bar=True)

# Store the embeddings as a NumPy array (already is a NumPy array by default)
print(f"Generated embeddings with shape: {law_embeddings.shape}")

# Display the first few embeddings as an example
print("\nFirst 5 embeddings:")
print(law_embeddings[:5])

## Build faiss index

### Subtask:
Create a FAISS index from the generated embeddings stored in the `law_embeddings` NumPy array.


**Reasoning**:
Create a FAISS index from the generated embeddings.



In [ ]:
import faiss

# Get the dimension of the embeddings
embedding_dimension = law_embeddings.shape[1]

# Create a FAISS index (FlatL2 index for exact nearest neighbor search)
index = faiss.IndexFlatL2(embedding_dimension)

# Add the law embeddings to the index
index.add(law_embeddings)

# Print the number of vectors added to the index
print(f"Number of vectors added to the index: {index.ntotal}")

## Implement retrieval function

### Subtask:
Update the `retrieve_laws` function to perform a similarity search on the FAISS index using the embedding of the user's question and return the text of the top-k most similar law articles.


**Reasoning**:
Update the `retrieve_laws` function to use the loaded text encoder model to embed the question, and then perform a similarity search on the FAISS index to retrieve the top-k most similar law articles from the processed DataFrame.



In [ ]:
import numpy as np # Already imported, but good to be explicit if using numpy functions

def retrieve_laws(question, k=5):
    """
    Retrieves relevant law articles based on the user's question using SBERT and FAISS.

    Args:
        question (str): The user's question.
        k (int): The number of top relevant articles to retrieve.

    Returns:
        list: A list of strings, where each string is the text of a retrieved law article.
              Returns an empty list if retrieval fails or no laws are found.
    """
    if 'text_encoder_model' not in globals() or text_encoder_model is None:
        print("Error: Text encoder model is not loaded.")
        return []
    if 'index' not in globals() or index is None:
        print("Error: FAISS index is not built.")
        return []
    if 'processed_law_df' not in globals() or processed_law_df is None:
        print("Error: Processed law DataFrame is not available.")
        return []

    try:
        # Encode the user's question into an embedding
        question_embedding = text_encoder_model.encode(question)
        # FAISS expects a 2D array, even for a single vector
        question_embedding = np.array([question_embedding])

        # Perform similarity search on the FAISS index
        # D is the distances, I is the indices
        distances, indices = index.search(question_embedding, k)

        # Retrieve the text of the relevant law articles using the indices
        # The indices are returned as a 2D array, even for a single query.
        # We are interested in the indices from the first row (index 0)
        relevant_article_indices = indices[0]

        # Filter out -1 indices, which indicate that fewer than k results were found
        relevant_article_indices = relevant_article_indices[relevant_article_indices != -1]

        # Select the corresponding rows from the processed_law_df
        retrieved_articles = processed_law_df.iloc[relevant_article_indices]

        # Extract the 'law_text' from the retrieved articles
        retrieved_law_texts = retrieved_articles['law_text'].tolist()

        print(f"Successfully retrieved {len(retrieved_law_texts)} relevant law articles.")

        return retrieved_law_texts

    except Exception as e:
        print(f"An error occurred during law retrieval: {e}")
        return []

# Example usage to test the updated function (replace with actual questions later)
test_question = "Quy định về tốc độ xe khi tham gia giao thông?"
retrieved_laws_example = retrieve_laws(test_question, k=3)

print(f"\nTest Question: '{test_question}'")
print("Retrieved Laws:")
if retrieved_laws_example:
    for i, law in enumerate(retrieved_laws_example):
        print(f"{i+1}. {law}")
else:
    print("No relevant laws retrieved.")


## Test retrieval function

### Subtask:
Test the updated `retrieve_laws` function with sample text and image inputs to ensure it returns relevant results.


**Reasoning**:
The `retrieve_laws` function has been updated to use the SBERT model and FAISS index. Now, test this function with sample text and image inputs as instructed to ensure it returns relevant results.



In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt # Import for displaying images if needed

# 1. Define a sample text question
sample_text_question = "Lái xe vượt đèn đỏ bị phạt bao nhiêu?"

# 2. Call retrieve_laws with the sample text question
print("--- Testing retrieve_laws with text input ---")
k_results = 5 # Desired number of results
retrieved_laws_text = retrieve_laws(sample_text_question, k=k_results)

# 3. Print the retrieved law articles from text input
print(f"\nSample Text Question: '{sample_text_question}'")
print(f"Retrieved Top {k_results} Laws:")
if retrieved_laws_text:
    for i, law in enumerate(retrieved_laws_text):
        print(f"{i+1}. {law}")
else:
    print("No relevant laws retrieved for text input.")

# 4. Choose a sample image file path
print("\n--- Testing retrieve_laws with image input (via OCR) ---")
# Ensure law_image_files list is available and not empty
if 'law_image_files' in globals() and law_image_files:
    # Using the first image from the law_db folder as an example
    sample_image_name = law_image_files[0]
    sample_image_path = os.path.join(law_images_path, sample_image_name)
    print(f"Using sample image: {sample_image_path}")

    # 5. Call extract_text_from_image with the sample image file path
    # Ensure extract_text_from_image is defined (from cell CsNFgNn5sBIJ)
    if 'extract_text_from_image' in globals():
        extracted_text = extract_text_from_image(sample_image_path)

        # 6. If text extraction is successful, call retrieve_laws
        if extracted_text:
            print(f"\nExtracted Text from Image: '{extracted_text}'")
            retrieved_laws_image = retrieve_laws(extracted_text, k=k_results)

            # 7. Print the retrieved law articles from image input
            print(f"\nRetrieved Top {k_results} Laws based on Extracted Text:")
            if retrieved_laws_image:
                for i, law in enumerate(retrieved_laws_image):
                    print(f"{i+1}. {law}")
            else:
                print("No relevant laws retrieved based on extracted text.")
        else:
            # Error handling for text extraction failure
            print(f"\nCould not extract text from image: {sample_image_path}. Skipping law retrieval for this image.")
            # You might want to display the image here to see if it's readable
            # try:
            #     img = Image.open(sample_image_path)
            #     display(img)
            # except Exception as e:
            #     print(f"Could not display image: {e}")

    else:
        # Error handling if extract_text_from_image is not defined
        print("Error: extract_text_from_image function not found. Cannot process image input.")
else:
    # Error handling if no image files are available
    print("\nSkipping image input test. No law image files found or law_image_files list is not available.")
    print("Please ensure Google Drive is mounted and the law database image folder path is correct.")


**Reasoning**:
The `retrieve_laws` function was successfully tested with a text input. However, the image input test failed because the OCR encountered an error loading the language data. This needs to be fixed to fully test the retrieval function with image input. The error message indicates a problem with the TESSDATA_PREFIX environment variable and loading the Vietnamese language data.



In [ ]:
# The previous attempt to extract text from image failed due to Tesseract configuration.
# Let's try setting the TESSDATA_PREFIX environment variable and installing the language data again.
import os

# Set TESSDATA_PREFIX to the default tessdata directory
os.environ['TESSDATA_PREFIX'] = '/usr/share/tesseract-ocr/4.00/tessdata'

# Try installing Vietnamese language data specifically
# In some environments, the general tesseract-ocr package might not include all languages.
!sudo apt update
!sudo apt install tesseract-ocr-vie

# Now, re-run the image input test part of the previous cell
print("\n--- Retrying Testing retrieve_laws with image input (via OCR) after Tesseract fix ---")

# Ensure law_image_files list is available and not empty
if 'law_image_files' in globals() and law_image_files:
    # Using the first image from the law_db folder as an example
    sample_image_name = law_image_files[0]
    sample_image_path = os.path.join(law_images_path, sample_image_name)
    print(f"Using sample image: {sample_image_path}")

    # Call extract_text_from_image with the sample image file path
    if 'extract_text_from_image' in globals():
        extracted_text = extract_text_from_image(sample_image_path)

        # If text extraction is successful, call retrieve_laws
        if extracted_text:
            print(f"\nExtracted Text from Image: '{extracted_text}'")
            k_results = 5
            retrieved_laws_image = retrieve_laws(extracted_text, k=k_results)

            # Print the retrieved law articles from image input
            print(f"\nRetrieved Top {k_results} Laws based on Extracted Text:")
            if retrieved_laws_image:
                for i, law in enumerate(retrieved_laws_image):
                    print(f"{i+1}. {law}")
            else:
                print("No relevant laws retrieved based on extracted text.")
        else:
            # Error handling for text extraction failure
            print(f"\nCould not extract text from image: {sample_image_path}. Skipping law retrieval for this image.")
            print("Please check the image and ensure Tesseract is configured correctly.")

    else:
        # Error handling if extract_text_from_image is not defined
        print("Error: extract_text_from_image function not found. Cannot process image input.")
else:
    # Error handling if no image files are available
    print("\nSkipping image input test. No law image files found or law_image_files list is not available.")
    print("Please ensure Google Drive is mounted and the law database image folder path is correct.")

## Integrate retrieval into rag pipeline

### Subtask:
Ensure the `rag_pipeline` function correctly calls the updated `retrieve_laws` function.


**Reasoning**:
Rerun the cell containing the `rag_pipeline` function and its test calls to ensure it now uses the updated `retrieve_laws` function and observes the output.



In [ ]:
# Placeholder for the retrieve_laws function
# This function will be implemented in a later step using the Text Encoder and FAISS index
# The updated retrieve_laws function should already be defined in a previous cell.
# To ensure it's used, we will redefine it here if needed, or just rely on the definition
# from the previous successful subtask. Let's assume the updated function is available
# in the global scope from the previous step.

# The retrieve_laws function is assumed to be updated and available here.
# We will just use the existing rag_pipeline definition and test calls.

def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        # Input is likely an image file path, perform OCR
        print(f"Image input detected: {input_data}. Performing OCR...")
        # Ensure extract_text_from_image is defined (from cell CsNFgNn5sBIJ)
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        # Input is a text question
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question (extracted text or original text)
    # This will now call the updated retrieve_laws function using SBERT and FAISS
    related_laws = retrieve_laws(question) # Call the updated retrieve_laws function
    context = "\n".join(related_laws)
    if not related_laws:
        # Handle case where no relevant laws are found
        print("Warning: No relevant laws retrieved for the question.")
        # You might want to provide a default answer or handle this case specifically
        answer = "Xin lỗi, tôi không tìm thấy thông tin liên quan trong cơ sở dữ liệu luật."
        confidence = 0.0
    else:
        # Step 3: Generate answer using the question and context
        # Ensure the generate_answer function is defined (from cell MrgWbpEjmvJs)
        if 'generate_answer' not in globals():
            return {"error": "generate_answer function not defined. Please run the cell that defines it."}
        answer = generate_answer(question, context)
        # Confidence score is a placeholder for now, can be improved later
        confidence = 0.9 # Placeholder confidence


    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": confidence  # Có thể tính từ similarity score của FAISS
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
# Check if model and tokenizer for generation are loaded before running the pipeline
if 'tokenizer' in globals() and tokenizer is not None and 'model' in globals() and model is not None:
    print("--- Testing RAG pipeline with text input ---")
    result_text = rag_pipeline(question_text)
    print("Kết quả (Text Input):", result_text)
else:
    print("Skipping text input demo. Language model for generation not loaded.")


# Demo with image input (using an example image from your Google Drive)
# Ensure you have run the Google Drive mounting and listing cells first
# and that law_image_files list is available and not empty
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0] # Use the first image from the law_db folder
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing RAG pipeline with image input: {example_image_path} ---")
    # Check if necessary functions are defined before attempting image processing
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")

**Reasoning**:
The previous execution failed with an `IndexError` in the `model.generate` call, specifically within the embedding layer. This indicates that the input `input_ids` to the model contain token IDs that are outside the vocabulary size of the model's embedding layer. This often happens when the input sequence (question + context) after tokenization and truncation exceeds the maximum sequence length the model can handle, or if there's an issue with how the context and question are combined and tokenized. I need to inspect the tokenization process and the resulting input IDs before passing them to the model to identify the problematic token ID.



In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM # Import AutoModelForCausalLM
import torch # Import torch

# Load model và tokenizer (re-loading to ensure they are available)
try:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    model = AutoModelForCausalLM.from_pretrained("vinai/phobert-base")
    print("Successfully loaded vinai/phobert-base model and tokenizer.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Could not load vinai/phobert-base as a CausalLM. Please check model compatibility or try a different model.")
    tokenizer, model = None, None # Set to None if loading fails

def generate_answer(question, context):
    if tokenizer is None or model is None:
        return "Error: Language model not loaded."

    prompt = f"""Context: {context}
    Câu hỏi: {question}
    Trả lời ngắn gọn dựa trên context:
    """

    # Tokenize the input
    # Add return_overflowing_tokens=True to inspect potential truncation issues
    # Add max_length and truncation=True to handle long inputs
    # Ensure padding='max_length' or 'longest' if batching (not doing batching here, but good practice)
    # Using return_tensors="pt" for PyTorch tensors
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=tokenizer.model_max_length, # Use the model's max length
        padding='max_length' # Pad to max_length
        # return_overflowing_tokens=True # Uncomment to inspect overflowing tokens
    )

    print("\n--- Debugging Tokenization ---")
    print(f"Input Prompt Length: {len(prompt)}")
    print(f"Tokenized Input IDs Shape: {inputs['input_ids'].shape}")
    print(f"Tokenized Input IDs (first 50): {inputs['input_ids'][0][:50]}")
    print(f"Tokenized Input IDs (last 50): {inputs['input_ids'][0][-50:]}")
    print(f"Maximum Token ID in input: {torch.max(inputs['input_ids'])}")
    print(f"Model Vocabulary Size: {tokenizer.vocab_size}")
    print("-----------------------------")

    # Check if any token ID is out of the vocabulary range
    if torch.max(inputs['input_ids']) >= tokenizer.vocab_size:
        print("Error: Token ID out of vocabulary range detected!")
        # Find the problematic token IDs
        problematic_ids = inputs['input_ids'][inputs['input_ids'] >= tokenizer.vocab_size]
        print(f"Problematic Token IDs: {problematic_ids}")
        # Decode some of the input to see what might be causing issues
        decoded_input = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=False)
        print(f"Decoded input (might show where problematic tokens are): {decoded_input[:500]}...") # Print first 500 chars
        return "Error: Input tokenization resulted in out-of-vocabulary IDs."


    # Generate the answer
    # Added parameters for potentially better generation with CausalLMs
    # Note: PhoBERT is not primarily a CausalLM for free-form generation.
    # The generation results might be poor or unexpected.
    # The previous error suggests an issue with how the model handles the input length/tokens.
    # Let's keep the generation parameters for now, but be aware of potential limitations.
    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=150, # Increased token limit
            num_return_sequences=1,
            # no_repeat_ngram_size=2, # Avoid repeating ngrams - might cause issues with small models
            # early_stopping=True, # Stop early
            # temperature=0.7, # Control randomness
            # top_k=50, # Limit to top K tokens
            # top_p=0.95 # Use nucleus sampling
            # Keep only essential generation parameters to simplify and see if error persists
            do_sample=False, # Use greedy decoding
            max_length=inputs['input_ids'].shape[1] + 150 # Set max_length for generation
        )

        # Decode the generated output
        # Decode only the newly generated tokens, excluding the prompt
        # Find the length of the input prompt tokens
        input_length = inputs['input_ids'].shape[1]
        # Decode from the end of the input tokens
        answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        # Basic post-processing
        answer = answer.strip()
        # Sometimes the prompt is repeated, try to remove a simplified version of it
        # This is a heuristic and might need adjustment
        simple_prompt_part = "Trả lời ngắn gọn dựa trên context:"
        if answer.startswith(simple_prompt_part):
             answer = answer[len(simple_prompt_part):].strip()


        return answer

    except Exception as e:
        print(f"An error occurred during generation: {e}")
        return "Error: Failed to generate answer."


# Redefine rag_pipeline to use the potentially fixed generate_answer
def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        # Input is likely an image file path, perform OCR
        print(f"Image input detected: {input_data}. Performing OCR...")
        # Ensure extract_text_from_image is defined (from cell CsNFgNn5sBIJ)
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        # Input is a text question
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question (extracted text or original text)
    related_laws = retrieve_laws(question) # Call the updated retrieve_laws function
    context = "\n".join(related_laws)
    if not related_laws:
        print("Warning: No relevant laws retrieved for the question.")
        answer = "Xin lỗi, tôi không tìm thấy thông tin liên quan trong cơ sở dữ liệu luật."
        confidence = 0.0
    else:
        # Step 3: Generate answer using the question and context
        if 'generate_answer' not in globals():
            return {"error": "generate_answer function not defined. Please run the cell that defines it."}
        answer = generate_answer(question, context)
        confidence = 0.9 # Placeholder confidence


    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": confidence
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
if 'tokenizer' in globals() and tokenizer is not None and 'model' in globals() and model is not None:
    print("--- Testing RAG pipeline with text input ---")
    result_text = rag_pipeline(question_text)
    print("Kết quả (Text Input):", result_text)
else:
    print("Skipping text input demo. Language model for generation not loaded.")


# Demo with image input (using an example image from your Google Drive)
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0]
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing RAG pipeline with image input: {example_image_path} ---")
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")


**Reasoning**:
The traceback indicates an `OverflowError` during tokenization within the `tokenizer.pad` method, specifically when trying to extend the attention mask list. This error suggests that the calculated padding amount (`difference`) or the resulting list length is extremely large, exceeding the capacity of a standard integer index. This is likely caused by an issue with `max_length` or the length of the input sequence after tokenization. The `max_length` was set to `tokenizer.model_max_length`, which might be a very large value for this specific model interpreted as a CausalLM, or there's a problem with the input prompt itself being excessively long. I need to inspect the length of the prompt and the tokenization inputs before padding to understand why this is happening. I will add prints to check the length of the raw prompt, the tokenized IDs before padding, and the calculated `max_length` being used for padding. I will also explicitly set a reasonable `max_length` for tokenization if `tokenizer.model_max_length` seems problematic.



In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
import torch
import os # Ensure os is imported for file path operations

# Load model và tokenizer (re-loading to ensure they are available)
try:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    # Use from_pretrained without specifying task like CausalLM initially
    # This might load the base model which might not have a generation head
    # but it helps debug tokenization independently.
    # If generation is needed later, we might need a different model or explicit head.
    model = AutoModelForCausalLM.from_pretrained("vinai/phobert-base")
    print("Successfully loaded vinai/phobert-base model and tokenizer.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Could not load vinai/phobert-base. Please check model compatibility or try a different model.")
    tokenizer, model = None, None

def generate_answer(question, context):
    if tokenizer is None or model is None:
        return "Error: Language model not loaded."

    prompt = f"""Context: {context}
    Câu hỏi: {question}
    Trả lời ngắn gọn dựa trên context:
    """

    print("\n--- Debugging Tokenization ---")
    print(f"Raw Prompt Length (characters): {len(prompt)}")

    # Tokenize the input without padding initially to check the length
    raw_inputs = tokenizer(prompt, return_tensors="pt", truncation=False, padding=False)
    print(f"Tokenized Input IDs (before padding) Shape: {raw_inputs['input_ids'].shape}")
    print(f"Tokenized Input IDs (before padding) Length: {raw_inputs['input_ids'].shape[1]}")

    # Check the model's maximum sequence length
    model_max_length = tokenizer.model_max_length
    print(f"Model's max_length according to tokenizer: {model_max_length}")

    # Set a reasonable max_length for tokenization and padding
    # If model_max_length is excessively large or -1 (meaning no inherent limit),
    # we should choose a practical limit, e.g., 512 or 1024 tokens.
    # Let's try a fixed max_length if the model_max_length is problematic.
    # PhoBERT base has a max length of 256 or 512 depending on configuration.
    # Let's use 512 as a common value.
    effective_max_length = 512 # A common sequence length for BERT-like models

    # Re-tokenize with truncation and padding to the effective_max_length
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=effective_max_length,
        padding='max_length'
    )

    print(f"Tokenized Input IDs (after padding and truncation) Shape: {inputs['input_ids'].shape}")
    print(f"Tokenized Input IDs (after padding and truncation) Length: {inputs['input_ids'].shape[1]}")
    print(f"Maximum Token ID in input: {torch.max(inputs['input_ids'])}")
    print(f"Model Vocabulary Size: {tokenizer.vocab_size}")
    print("-----------------------------")

    # Check if any token ID is out of the vocabulary range (should not happen with standard tokenizers unless corrupted)
    if torch.max(inputs['input_ids']) >= tokenizer.vocab_size:
        print("Error: Token ID out of vocabulary range detected AFTER re-tokenization!")
        return "Error: Input tokenization resulted in out-of-vocabulary IDs after re-tokenization."


    # Generate the answer
    try:
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_length=inputs['input_ids'].shape[1] + 150 # Set max_length for generation
        )

        # Decode the generated output
        input_length = inputs['input_ids'].shape[1]
        answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        # Basic post-processing
        answer = answer.strip()
        simple_prompt_part = "Trả lời ngắn gọn dựa trên context:"
        if answer.startswith(simple_prompt_part):
             answer = answer[len(simple_prompt_part):].strip()

        return answer

    except Exception as e:
        print(f"An error occurred during generation: {e}")
        return "Error: Failed to generate answer."


# Redefine rag_pipeline to use the potentially fixed generate_answer
def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        print(f"Image input detected: {input_data}. Performing OCR...")
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question
    if 'retrieve_laws' not in globals():
         return {"error": "retrieve_laws function not defined. Please run the cell that defines it."}
    related_laws = retrieve_laws(question)
    context = "\n".join(related_laws)
    if not related_laws:
        print("Warning: No relevant laws retrieved for the question.")
        answer = "Xin lỗi, tôi không tìm thấy thông tin liên quan trong cơ sở dữ liệu luật."
        confidence = 0.0
    else:
        # Step 3: Generate answer using the question and context
        if 'generate_answer' not in globals():
            return {"error": "generate_answer function not defined. Please run the cell that defines it."}
        answer = generate_answer(question, context)
        confidence = 0.9


    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": confidence
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
if 'tokenizer' in globals() and tokenizer is not None and 'model' in globals() and model is not None:
    print("--- Testing RAG pipeline with text input ---")
    result_text = rag_pipeline(question_text)
    print("Kết quả (Text Input):", result_text)
else:
    print("Skipping text input demo. Language model for generation not loaded.")


# Demo with image input
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0]
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing RAG pipeline with image input: {example_image_path} ---")
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")

**Reasoning**:
The traceback now shows an `OverflowError` occurring *before* the generation call, during the `tokenizer` call itself, specifically in the padding logic when adding to the attention mask. This confirms that the issue is still related to the size of the input sequence being processed by the tokenizer's padding function. Although setting `max_length=512` and `truncation=True` was intended to prevent this, the error suggests an internal issue within the tokenizer's padding implementation when dealing with the resulting list sizes. The debugging prints show the raw prompt length is over 5000 characters and the tokenized length before padding is over 1500 tokens. This is significantly longer than the 512 `max_length` we intended to use. The fact that it still tries to perform list addition with a large difference indicates a potential problem with how `truncation=True` and `padding='max_length'` interact or are interpreted in this specific tokenizer version or environment.

I will try to work around this by manually handling the truncation and padding. I will tokenize the prompt with `truncation=False` and `padding=False`, then manually truncate the resulting token IDs to `effective_max_length` (512), and then manually pad the truncated sequence to `effective_max_length`. This bypasses the potentially problematic internal padding logic in the tokenizer when dealing with very long initial sequences. I will also explicitly add the special tokens after manual truncation if the tokenizer's default behavior includes them at the beginning/end.



In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
import torch
import os # Ensure os is imported for file path operations

# Load model và tokenizer (re-loading to ensure they are available)
try:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    model = AutoModelForCausalLM.from_pretrained("vinai/phobert-base")
    print("Successfully loaded vinai/phobert-base model and tokenizer.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Could not load vinai/phobert-base. Please check model compatibility or try a different model.")
    tokenizer, model = None, None

def generate_answer(question, context):
    if tokenizer is None or model is None:
        return "Error: Language model not loaded."

    prompt = f"""Context: {context}
    Câu hỏi: {question}
    Trả lời ngắn gọn dựa trên context:
    """

    print("\n--- Debugging Tokenization ---")
    print(f"Raw Prompt Length (characters): {len(prompt)}")

    # Manually handle tokenization, truncation, and padding
    effective_max_length = 512
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id # Use pad_token_id or eos_token_id as fallback
    if pad_token_id is None:
         print("Error: Could not determine a padding token ID.")
         return "Error: Tokenizer has no pad or eos token ID."


    # Tokenize without truncation or padding first
    raw_tokens = tokenizer.encode(prompt, add_special_tokens=False) # Encode without adding special tokens initially

    print(f"Tokenized Input IDs (before manual processing) Length: {len(raw_tokens)}")
    print(f"Model's max_length according to tokenizer: {tokenizer.model_max_length}")
    print(f"Effective max_length used: {effective_max_length}")

    # Manually truncate if longer than effective_max_length
    if len(raw_tokens) > effective_max_length:
        print(f"Truncating token IDs from {len(raw_tokens)} to {effective_max_length}")
        truncated_tokens = raw_tokens[:effective_max_length]
    else:
        truncated_tokens = raw_tokens

    # Manually add special tokens (e.g., CLS and SEP for BERT-like models)
    # PhoBERT uses <s> and </s>
    # Let's manually add them, ensuring we respect the max_length after adding
    # Reserve space for special tokens: <s> + tokens + </s>
    cls_token_id = tokenizer.cls_token_id # <s>
    sep_token_id = tokenizer.sep_token_id # </s>

    # Ensure we have space for special tokens if adding
    space_for_special = 0
    if cls_token_id is not None: space_for_special += 1
    if sep_token_id is not None: space_for_special += 1


    # Re-truncate if adding special tokens would exceed max_length
    if len(truncated_tokens) > effective_max_length - space_for_special:
         truncated_tokens = truncated_tokens[:effective_max_length - space_for_special]
         print(f"Further truncating for special tokens to {effective_max_length - space_for_special}")


    final_tokens = []
    if cls_token_id is not None:
         final_tokens.append(cls_token_id)
    final_tokens.extend(truncated_tokens)
    if sep_token_id is not None:
         final_tokens.append(sep_token_token_id) # Corrected variable name


    # Manually pad to effective_max_length
    padding_length = effective_max_length - len(final_tokens)
    if padding_length > 0:
        print(f"Padding token IDs with {padding_length} tokens using ID {pad_token_id}")
        padded_tokens = final_tokens + [pad_token_id] * padding_length
    else:
        padded_tokens = final_tokens # Should be exactly effective_max_length if truncated

    # Convert to PyTorch tensor
    input_ids = torch.tensor([padded_tokens]) # Add batch dimension

    # Create attention mask manually (1 for real tokens, 0 for padding)
    attention_mask = torch.tensor([[1] * len(final_tokens) + [0] * padding_length])


    # Create token type IDs (all zeros for single sequence input)
    token_type_ids = torch.zeros_like(input_ids)


    inputs = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'token_type_ids': token_type_ids # Include token_type_ids as PhoBERT uses them
    }


    print(f"Tokenized Input IDs (after manual processing) Shape: {inputs['input_ids'].shape}")
    print(f"Tokenized Input IDs (after manual processing) Length: {inputs['input_ids'].shape[1]}")
    print(f"Maximum Token ID in input: {torch.max(inputs['input_ids'])}")
    print(f"Model Vocabulary Size: {tokenizer.vocab_size}")
    # print(f"Input IDs: {inputs['input_ids']}") # Uncomment to see the actual IDs
    # print(f"Attention Mask: {inputs['attention_mask']}") # Uncomment to see the actual mask
    # print(f"Token Type IDs: {inputs['token_type_ids']}") # Uncomment to see the actual token type IDs
    print("-----------------------------")


    # Check if any token ID is out of the vocabulary range
    if torch.max(inputs['input_ids']) >= tokenizer.vocab_size:
        print("Error: Token ID out of vocabulary range detected AFTER manual processing!")
        return "Error: Input tokenization resulted in out-of-vocabulary IDs after manual processing."


    # Generate the answer
    try:
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=inputs['token_type_ids'], # Pass token_type_ids
            do_sample=False,
            max_length=inputs['input_ids'].shape[1] + 150 # Set max_length for generation
        )

        # Decode the generated output
        input_length = inputs['input_ids'].shape[1]
        # Ensure outputs tensor is not empty and has the expected shape before slicing
        if outputs.shape[1] > input_length:
             answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        else:
             print("Warning: Generation did not produce new tokens.")
             answer = "" # Or some default message


        # Basic post-processing
        answer = answer.strip()
        simple_prompt_part = "Trả lời ngắn gọn dựa trên context:"
        if answer.startswith(simple_prompt_part):
             answer = answer[len(simple_prompt_part):].strip()

        return answer

    except Exception as e:
        print(f"An error occurred during generation: {e}")
        return "Error: Failed to generate answer."


# Redefine rag_pipeline to use the potentially fixed generate_answer
def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        print(f"Image input detected: {input_data}. Performing OCR...")
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question
    if 'retrieve_laws' not in globals():
         return {"error": "retrieve_laws function not defined. Please run the cell that defines it."}
    related_laws = retrieve_laws(question)
    context = "\n".join(related_laws)
    if not related_laws:
        print("Warning: No relevant laws retrieved for the question.")
        answer = "Xin lỗi, tôi không tìm thấy thông tin liên quan trong cơ sở dữ liệu luật."
        confidence = 0.0
    else:
        # Step 3: Generate answer using the question and context
        if 'generate_answer' not in globals():
            return {"error": "generate_answer function not defined. Please run the cell that defines it."}
        answer = generate_answer(question, context)
        confidence = 0.9


    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": confidence
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
if 'tokenizer' in globals() and tokenizer is not None and 'model' in globals() and model is not None:
    print("--- Testing RAG pipeline with text input ---")
    result_text = rag_pipeline(question_text)
    print("Kết quả (Text Input):", result_text)
else:
    print("Skipping text input demo. Language model for generation not loaded.")


# Demo with image input
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0]
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing RAG pipeline with image input: {example_image_path} ---")
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")

**Reasoning**:
The traceback shows a `NameError: name 'sep_token_token_id' is not defined`. This is a simple typo; the variable should be `sep_token_id`. I need to fix this typo in the `generate_answer` function and then re-run the cell to continue testing the RAG pipeline with the manual tokenization, truncation, and padding logic.



In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
import torch
import os

# Load model và tokenizer (re-loading to ensure they are available)
try:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    model = AutoModelForCausalLM.from_pretrained("vinai/phobert-base")
    print("Successfully loaded vinai/phobert-base model and tokenizer.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Could not load vinai/phobert-base. Please check model compatibility or try a different model.")
    tokenizer, model = None, None

def generate_answer(question, context):
    if tokenizer is None or model is None:
        return "Error: Language model not loaded."

    prompt = f"""Context: {context}
    Câu hỏi: {question}
    Trả lời ngắn gọn dựa trên context:
    """

    print("\n--- Debugging Tokenization ---")
    print(f"Raw Prompt Length (characters): {len(prompt)}")

    # Manually handle tokenization, truncation, and padding
    effective_max_length = 512
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    if pad_token_id is None:
         print("Error: Could not determine a padding token ID.")
         return "Error: Tokenizer has no pad or eos token ID."


    # Tokenize without truncation or padding first
    raw_tokens = tokenizer.encode(prompt, add_special_tokens=False)

    print(f"Tokenized Input IDs (before manual processing) Length: {len(raw_tokens)}")
    print(f"Model's max_length according to tokenizer: {tokenizer.model_max_length}")
    print(f"Effective max_length used: {effective_max_length}")

    # Manually truncate if longer than effective_max_length
    if len(raw_tokens) > effective_max_length:
        print(f"Truncating token IDs from {len(raw_tokens)} to {effective_max_length}")
        truncated_tokens = raw_tokens[:effective_max_length]
    else:
        truncated_tokens = raw_tokens

    # Manually add special tokens (e.g., CLS and SEP for BERT-like models)
    # PhoBERT uses <s> and </s>
    cls_token_id = tokenizer.cls_token_id # <s>
    sep_token_id = tokenizer.sep_token_id # </s>

    # Ensure we have space for special tokens if adding
    space_for_special = 0
    if cls_token_id is not None: space_for_special += 1
    if sep_token_id is not None: space_for_special += 1

    # Re-truncate if adding special tokens would exceed max_length
    if len(truncated_tokens) > effective_max_length - space_for_special:
         truncated_tokens = truncated_tokens[:effective_max_length - space_for_special]
         print(f"Further truncating for special tokens to {effective_max_length - space_for_special}")


    final_tokens = []
    if cls_token_id is not None:
         final_tokens.append(cls_token_id)
    final_tokens.extend(truncated_tokens)
    if sep_token_id is not None:
         final_tokens.append(sep_token_id) # Corrected the typo here


    # Manually pad to effective_max_length
    padding_length = effective_max_length - len(final_tokens)
    if padding_length > 0:
        print(f"Padding token IDs with {padding_length} tokens using ID {pad_token_id}")
        padded_tokens = final_tokens + [pad_token_id] * padding_length
    else:
        padded_tokens = final_tokens

    # Convert to PyTorch tensor
    input_ids = torch.tensor([padded_tokens])

    # Create attention mask manually (1 for real tokens, 0 for padding)
    attention_mask = torch.tensor([[1] * len(final_tokens) + [0] * padding_length])


    # Create token type IDs (all zeros for single sequence input)
    token_type_ids = torch.zeros_like(input_ids)


    inputs = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'token_type_ids': token_type_ids
    }


    print(f"Tokenized Input IDs (after manual processing) Shape: {inputs['input_ids'].shape}")
    print(f"Tokenized Input IDs (after manual processing) Length: {inputs['input_ids'].shape[1]}")
    print(f"Maximum Token ID in input: {torch.max(inputs['input_ids'])}")
    print(f"Model Vocabulary Size: {tokenizer.vocab_size}")
    print("-----------------------------")


    # Check if any token ID is out of the vocabulary range
    if torch.max(inputs['input_ids']) >= tokenizer.vocab_size:
        print("Error: Token ID out of vocabulary range detected AFTER manual processing!")
        return "Error: Input tokenization resulted in out-of-vocabulary IDs after manual processing."


    # Generate the answer
    try:
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            token_type_ids=inputs['token_type_ids'],
            do_sample=False,
            max_length=inputs['input_ids'].shape[1] + 150
        )

        # Decode the generated output
        input_length = inputs['input_ids'].shape[1]
        if outputs.shape[1] > input_length:
             answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        else:
             print("Warning: Generation did not produce new tokens.")
             answer = ""


        # Basic post-processing
        answer = answer.strip()
        simple_prompt_part = "Trả lời ngắn gọn dựa trên context:"
        if answer.startswith(simple_prompt_part):
             answer = answer[len(simple_prompt_part):].strip()

        return answer

    except Exception as e:
        print(f"An error occurred during generation: {e}")
        return "Error: Failed to generate answer."


# Redefine rag_pipeline to use the potentially fixed generate_answer
def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        print(f"Image input detected: {input_data}. Performing OCR...")
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question
    if 'retrieve_laws' not in globals():
         return {"error": "retrieve_laws function not defined. Please run the cell that defines it."}
    related_laws = retrieve_laws(question)
    context = "\n".join(related_laws)
    if not related_laws:
        print("Warning: No relevant laws retrieved for the question.")
        answer = "Xin lỗi, tôi không tìm thấy thông tin liên quan trong cơ sở dữ liệu luật."
        confidence = 0.0
    else:
        # Step 3: Generate answer using the question and context
        if 'generate_answer' not in globals():
            return {"error": "generate_answer function not defined. Please run the cell that defines it."}
        answer = generate_answer(question, context)
        confidence = 0.9


    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": confidence
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
if 'tokenizer' in globals() and tokenizer is not None and 'model' in globals() and model is not None:
    print("--- Testing RAG pipeline with text input ---")
    result_text = rag_pipeline(question_text)
    print("Kết quả (Text Input):", result_text)
else:
    print("Skipping text input demo. Language model for generation not loaded.")


# Demo with image input
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0]
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing RAG pipeline with image input: {example_image_path} ---")
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")

**Reasoning**:
The `NameError` from the typo has been fixed, but the code still fails with an `index out of range in self` error during the `model.generate` call, even with manual tokenization and padding to 512 tokens. This error is similar to the initial `IndexError` and suggests that the model is receiving input token IDs that are outside of its expected range, possibly related to the position embeddings or token embeddings. The debugging prints show that the maximum token ID in the input (63117) is within the vocabulary size (64000), so it's not a simple out-of-vocabulary issue. PhoBERT is a BERT-like model, typically used for classification or token tasks, not free-form text generation as a CausalLM without a specific generation head. Loading it with `AutoModelForCausalLM` might be causing compatibility issues or expecting input/output formats that the base PhoBERT model isn't designed for in a generation context.

Given the repeated `index out of range` error during generation with the PhoBERT model, and its nature as a BERT-like model not primarily for causal language modeling, it's likely that this model is not suitable for the text generation task in this RAG pipeline using the current approach. The error persists even after attempting to fix tokenization and input formatting.

To address this, I will acknowledge that the current model (`vinai/phobert-base`) seems unsuitable for the generation task in this configuration. I will modify the `generate_answer` function to return a placeholder error message if the model is loaded, indicating that generation with this model is not working as expected. This allows the `rag_pipeline` to complete its execution flow without crashing at the generation step, demonstrating that the retrieval part works and the pipeline structure is functional, even if the generation component needs a different model or approach. I will also add a note suggesting that a different model specifically designed for text generation (like a small GPT, BART, or T5 fine-tuned for Vietnamese) would be more appropriate for this task.

This step focuses on making the pipeline runnable and identifying the current bottleneck (the generation model) without attempting to fix the generation capabilities of an unsuitable model.



In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM
import torch
import os

# Load model và tokenizer (re-loading to ensure they are available)
try:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    # Loading as AutoModelForCausalLM still causes issues.
    # Let's load it as AutoModel to avoid assuming a generation head,
    # but acknowledge it's not designed for text generation out-of-the-box this way.
    # The previous errors strongly suggest this model is not working for generation.
    # We will handle this by returning an error in generate_answer.
    model = AutoModelForCausalLM.from_pretrained("vinai/phobert-base") # Keep this for now to match structure, but handle error below
    print("Successfully loaded vinai/phobert-base model and tokenizer.")
    model_loaded_for_generation = True # Flag to indicate model was loaded
except Exception as e:
    print(f"Error loading model: {e}")
    print("Could not load vinai/phobert-base for generation. A different model might be needed.")
    tokenizer, model = None, None
    model_loaded_for_generation = False # Flag to indicate model loading failed


def generate_answer(question, context):
    # Check the flag instead of just checking if model is None
    if not model_loaded_for_generation or tokenizer is None or model is None:
        print("Error: Language model for generation is not loaded or not suitable.")
        return "Error: Generation model not available or suitable."

    prompt = f"""Context: {context}
    Câu hỏi: {question}
    Trả lời ngắn gọn dựa trên context:
    """

    print("\n--- Debugging Tokenization ---")
    print(f"Raw Prompt Length (characters): {len(prompt)}")

    # Manually handle tokenization, truncation, and padding
    effective_max_length = 512
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    if pad_token_id is None:
         print("Error: Could not determine a padding token ID.")
         return "Error: Tokenizer has no pad or eos token ID."

    try:
        # Tokenize without truncation or padding first
        raw_tokens = tokenizer.encode(prompt, add_special_tokens=False)

        print(f"Tokenized Input IDs (before manual processing) Length: {len(raw_tokens)}")
        print(f"Model's max_length according to tokenizer: {tokenizer.model_max_length}")
        print(f"Effective max_length used: {effective_max_length}")

        # Manually truncate if longer than effective_max_length
        if len(raw_tokens) > effective_max_length:
            print(f"Truncating token IDs from {len(raw_tokens)} to {effective_max_length}")
            truncated_tokens = raw_tokens[:effective_max_length]
        else:
            truncated_tokens = raw_tokens

        # Manually add special tokens (e.g., CLS and SEP for BERT-like models)
        # PhoBERT uses <s> (CLS) and </s> (SEP)
        cls_token_id = tokenizer.cls_token_id # <s>
        sep_token_id = tokenizer.sep_token_id # </s>

        # Ensure we have space for special tokens if adding
        space_for_special = 0
        if cls_token_id is not None: space_for_special += 1
        if sep_token_id is not None: space_for_special += 1

        # Re-truncate if adding special tokens would exceed max_length
        if len(truncated_tokens) > effective_max_length - space_for_special:
             truncated_tokens = truncated_tokens[:effective_max_length - space_for_special]
             print(f"Further truncating for special tokens to {effective_max_length - space_for_special}")


        final_tokens = []
        if cls_token_id is not None:
             final_tokens.append(cls_token_id)
        final_tokens.extend(truncated_tokens)
        if sep_token_id is not None:
             final_tokens.append(sep_token_id)


        # Manually pad to effective_max_length
        padding_length = effective_max_length - len(final_tokens)
        if padding_length > 0:
            print(f"Padding token IDs with {padding_length} tokens using ID {pad_token_id}")
            padded_tokens = final_tokens + [pad_token_id] * padding_length
        else:
            padded_tokens = final_tokens

        # Convert to PyTorch tensor
        input_ids = torch.tensor([padded_tokens])

        # Create attention mask manually (1 for real tokens, 0 for padding)
        attention_mask = torch.tensor([[1] * len(final_tokens) + [0] * padding_length])

        # Create token type IDs (all zeros for single sequence input)
        token_type_ids = torch.zeros_like(input_ids)

        inputs = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'token_type_ids': token_type_ids
        }

        print(f"Tokenized Input IDs (after manual processing) Shape: {inputs['input_ids'].shape}")
        print(f"Tokenized Input IDs (after manual processing) Length: {inputs['input_ids'].shape[1]}")
        print(f"Maximum Token ID in input: {torch.max(inputs['input_ids'])}")
        print(f"Model Vocabulary Size: {tokenizer.vocab_size}")
        print("-----------------------------")

        # Check if any token ID is out of the vocabulary range
        if torch.max(inputs['input_ids']) >= tokenizer.vocab_size:
            print("Error: Token ID out of vocabulary range detected AFTER manual processing!")
            return "Error: Input tokenization resulted in out-of-vocabulary IDs after manual processing."

    except Exception as e:
        print(f"An error occurred during tokenization or manual processing: {e}")
        return "Error: Failed during tokenization or manual processing."


    # --- Placeholder for actual generation ---
    # Based on previous errors, vinai/phobert-base loaded as CausalLM is not working for generation.
    print("Skipping actual generation with vinai/phobert-base due to known issues.")
    print("Please consider using a different model specifically designed for text generation.")
    generated_answer = "Placeholder: Answer generation failed with the current model. A suitable generation model is needed."
    # --- End Placeholder ---

    return generated_answer


# Redefine rag_pipeline to use the potentially fixed generate_answer (which now returns a placeholder)
def rag_pipeline(input_data):
    """
    Processes input (text question or image path) through the RAG pipeline.

    Args:
        input_data (str): User's question (text) or path to an image file.

    Returns:
        dict: A dictionary containing the generated answer, related laws, and confidence score.
    """
    # Step 1: Handle input (text or image)
    if isinstance(input_data, str) and os.path.isfile(input_data):
        print(f"Image input detected: {input_data}. Performing OCR...")
        if 'extract_text_from_image' not in globals():
             return {"error": "extract_text_from_image function not defined. Please run the cell that defines it."}
        question = extract_text_from_image(input_data)
        if not question:
            return {"error": f"Could not extract text from image {input_data}. Cannot proceed with retrieval."}
        print(f"Extracted text from image: '{question}'")
    elif isinstance(input_data, str):
        question = input_data
        print(f"Text input detected: '{question}'")
    else:
        return {"error": "Invalid input type. Please provide a text question or an image file path."}

    # Step 2: Retrieve relevant laws based on the question
    if 'retrieve_laws' not in globals():
         return {"error": "retrieve_laws function not defined. Please run the cell that defines it."}
    related_laws = retrieve_laws(question)
    context = "\n".join(related_laws)
    if not related_laws:
        print("Warning: No relevant laws retrieved for the question.")
        answer = "Xin lỗi, tôi không tìm thấy thông tin liên quan trong cơ sở dữ liệu luật."
        confidence = 0.0
    else:
        # Step 3: Generate answer using the question and context
        if 'generate_answer' not in globals():
            return {"error": "generate_answer function not defined. Please run the cell that defines it."}
        # This will call the generate_answer function which now returns a placeholder
        answer = generate_answer(question, context)
        # Confidence is a placeholder, would ideally be based on retrieval and generation
        confidence = 0.9


    return {
        "answer": answer,
        "related_laws": related_laws,
        "confidence": confidence
    }

# Demo with text input
question_text = "Xe máy có được đi vào đường cấm xe tải không?"
# The rag_pipeline now handles the model availability check internally via generate_answer
print("--- Testing RAG pipeline with text input ---")
result_text = rag_pipeline(question_text)
print("Kết quả (Text Input):", result_text)


# Demo with image input
if 'law_image_files' in globals() and law_image_files:
    example_image_name = law_image_files[0]
    example_image_path = os.path.join(law_images_path, example_image_name)
    print(f"\n--- Testing RAG pipeline with image input: {example_image_path} ---")
    if 'preprocess_image' in globals() and 'extract_text_from_image' in globals():
        result_image = rag_pipeline(example_image_path)
        print("Kết quả (Image Input):", result_image)
    else:
        print("Skipping image input demo. preprocess_image or extract_text_from_image function not found.")
else:
    print("\nSkipping image input demo. No law image files found or law_image_files list is not available.")

## Summary:

### Data Analysis Key Findings

*   The "keepitreal/vietnamese-sbert" model was successfully loaded and used to generate 768-dimensional embeddings for 399 law articles.
*   A FAISS `IndexFlatL2` was successfully created from these embeddings, containing 399 vectors.
*   The `retrieve_laws` function was updated to utilize the SBERT model for question embedding and the FAISS index for efficient similarity search, returning the top-k most relevant law article texts.
*   Testing confirmed the `retrieve_laws` function successfully retrieves relevant laws for text input queries.
*   The integration into the `rag_pipeline` was verified, showing that retrieved laws are passed as context to the answer generation step.
*   The chosen generation model ("vinai/phobert-base" loaded as `AutoModelForCausalLM`) was found to be unsuitable for the text generation task, leading to errors during the generation step, despite successful input processing up to that point.

### Insights or Next Steps

*   Replace the current generation model ("vinai/phobert-base") with a model specifically designed for text generation (e.g., a fine-tuned GPT-like model or a Seq2Seq model suitable for Vietnamese) to enable the RAG pipeline to produce meaningful answers based on the retrieved context.
*   Improve the robustness of the image processing pipeline, potentially by exploring alternative OCR engines or pre-processing techniques, as the current OCR quality hinders the effectiveness of image-based retrieval tests.


# 1. Chuẩn Bị Dữ Liệu Test
Từ file ảnh chụp, ta có:

📁 public_test/

├── 📁 public_test_images/        # Thư mục chứa ảnh biển báo test

├── 📄 vlsp_2025_public_test_task1.json (10KB)  # Câu hỏi dạng text

└── 📄 vlsp_2025_public_test_task2.json (35KB)  # Câu hỏi đa phương thức (text + ảnh)



Bước 1: Load Dữ Liệu Test

In [ ]:
import json
from PIL import Image
import os

# Đường dẫn đến thư mục ảnh test
TEST_IMAGES_DIR = "data/local/public_test/public_test_images"

# Lấy danh sách ảnh test
test_images = [f for f in os.listdir(TEST_IMAGES_DIR) if f.endswith(('.jpg', '.png'))]

# Load test task 1 (text-only)
with open('data/local/public_test/vlsp_2025_public_test_task1.json', 'r', encoding='utf-8') as f:
    test_task1 = json.load(f)  # List các dict: [{"question": "...", "correct_answer": "..."}, ...]

# Load test task 2 (đa phương thức)
with open('data/local/public_test/vlsp_2025_public_test_task2.json', 'r', encoding='utf-8') as f:
    test_task2 = json.load(f)  # Có thể chứa cả path đến ảnh

In [ ]:
# The main evaluation function is in cell IKMrDfWHzAtJ.
# This cell previously contained an incomplete evaluation function.
# Please refer to cell IKMrDfWHzAtJ for the updated evaluation logic.

Bước 2: Viết Hàm Đánh Giá

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
# Import necessary libraries for image processing if needed for multimodal evaluation
from PIL import Image
import pytesseract
import os

def evaluate(test_data, task_type='text'):
    predictions = []
    ground_truth = []
    # Store IDs for easier debugging
    test_item_ids = []

    for i, item in enumerate(test_data):
        question = None
        image_path = None
        extracted_text_from_image = ""

        # Capture item ID if available
        item_id = item.get('id', f'item_{i}')
        test_item_ids.append(item_id)

        # Handle input based on task type
        if task_type == 'text':
            if 'question' in item:
                question = item['question']
            if 'correct_answer' in item: # Check for correct_answer key (based on previous attempts)
                 ground_truth_answer = item['correct_answer']
            elif 'relevant_articles' in item: # Check for relevant_articles key
                 # For evaluation based on retrieval, you might compare retrieved articles
                 # with the 'relevant_articles' list. For now, we'll just store them.
                 # The actual comparison logic would need to be implemented here.
                 ground_truth_answer = item['relevant_articles'] # Storing the list of relevant articles as ground truth
                 # Note: Comparing generated text to a list of articles requires custom logic.
                 # Simple string comparison or f1_score won't work directly.
            else:
                 print(f"Warning: Neither 'correct_answer' nor 'relevant_articles' key found in item {item_id} for text task. Skipping item.")
                 print(f"Item keys: {item.keys()}") # Print keys to help diagnose
                 continue # Skip this item if ground truth is not found


        else:  # Multimodal task (text + image)
            if 'question' in item:
                question = item['question']
            if 'image_path' in item:
                # Construct the full path to the image
                image_base_path = 'data/local/public_test/public_test_images'
                image_path_relative = item['image_path']
                image_path = os.path.join(image_base_path, image_path_relative)

                # Perform OCR if image processing functions are available and file exists
                if 'extract_text_from_image' in globals() and os.path.exists(image_path):
                    print(f"Processing image: {image_path} for item {item_id}")
                    extracted_text_from_image = extract_text_from_image(image_path)
                    if not extracted_text_from_image:
                        print(f"Warning: Could not extract text from image {image_path} for item {item_id}. Using text question only.")
                elif not os.path.exists(image_path):
                     print(f"Warning: Image file not found at {image_path} for item {item_id}. Cannot process image input.")
                else:
                    print("Warning: extract_text_from_image function not found. Cannot process image input.")
                    image_path = None # Treat as text-only if image processing is not possible

            # Combine text question and extracted text from image (if any)
            if question and extracted_text_from_image:
                # You might want to structure this combined input differently depending on your RAG input
                question_for_rag = f"{question} (Thông tin từ ảnh: {extracted_text_from_image})"
            elif extracted_text_from_image:
                 question_for_rag = f"(Thông tin từ ảnh: {extracted_text_from_image})"
            elif question:
                 question_for_rag = question # Use original text question if no image or text extraction failed
            else:
                print(f"Warning: No text question or usable image found for item {item_id} in multimodal task. Skipping item.")
                continue # Skip if no question can be formed


            if 'correct_answer' in item: # Check for correct_answer key
                 ground_truth_answer = item['correct_answer']
            elif 'relevant_articles' in item: # Check for relevant_articles key
                 ground_truth_answer = item['relevant_articles'] # Storing the list of relevant articles as ground truth
            else:
                 print(f"Warning: Neither 'correct_answer' nor 'relevant_articles' key found in item {item_id} for multimodal task. Skipping item.")
                 print(f"Item keys: {item.keys()}") # Print keys to help diagnose
                 continue # Skip this item if ground truth is not found


        # Ensure a question was successfully formed for RAG
        if question_for_rag is None or question_for_rag.strip() == "":
             print(f"Warning: Could not form a valid question for RAG for item {item_id}. Skipping item.")
             continue


        # Call RAG pipeline if it's defined
        if 'rag_pipeline' in globals():
            # Pass the combined question (text + image info if multimodal) to the pipeline
            result = rag_pipeline(question_for_rag)
            # Extract the predicted answer from the result.
            # This is still a placeholder as generate_answer is a placeholder.
            predicted_answer = result.get('answer', '')
            predictions.append(predicted_answer)
            ground_truth.append(ground_truth_answer) # Use the retrieved correct answer or relevant articles

        else:
            print("Error: rag_pipeline function not found. Cannot perform evaluation.")
            return # Exit evaluation if pipeline is not available


    # Check if there are any predictions to evaluate
    if not predictions or not ground_truth:
        print("No valid test items processed. Cannot calculate metrics.")
        return


    # Calculate metrics
    # Note: accuracy_score and f1_score are suitable if correct_answer is a simple category or exact text match is expected.
    # If ground_truth is a list of articles, you need custom logic for comparison.
    # For generated text, more advanced metrics like ROUGE or BLEU might be needed.
    try:
        # --- IMPORTANT: Evaluation Logic Needs Customization ---
        # The following metrics (accuracy_score, f1_score) are likely NOT appropriate
        # if your ground truth is a list of relevant articles or if your generated
        # answer is free-form text that needs semantic comparison.

        # If ground truth is a single string correct_answer and prediction is generated text:
        # You would need a function to compare generated_text vs correct_answer (e.g., based on keywords, BLEU, ROUGE)
        # Example (pseudo-code):
        # matched_count = sum(compare_generated_answer(pred, truth) for pred, truth in zip(predictions, ground_truth))
        # accuracy = matched_count / len(predictions)

        # If ground truth is a list of relevant_articles and prediction is the generated answer text:
        # You might evaluate how well the generated answer reflects the content of the relevant articles.
        # This also requires custom logic or metrics.

        # If ground truth is a list of relevant_articles and prediction is the list of RETRIEVED articles:
        # You could evaluate retrieval performance using metrics like Precision, Recall, F1, MRR, NDCG
        # based on how many of the 'relevant_articles' were included in the 'related_laws' returned by rag_pipeline.
        # This would require capturing the 'related_laws' from the rag_pipeline result.

        # For now, we will print a warning and skip metric calculation because the current
        # ground truth format ('relevant_articles' list) is not compatible with simple metrics.
        print("\n--- Skipping Metric Calculation ---")
        print("Evaluation metrics (Accuracy, F1 Score) are not directly applicable to the current ground truth format (list of relevant articles).")
        print("Please implement custom evaluation logic or use metrics suitable for comparing generated text or evaluating retrieval against relevant articles.")
        print("--- End Skipping ---")

        # Example placeholder for how you *might* calculate a retrieval-based metric
        # This requires capturing the retrieved_laws from rag_pipeline result.
        # retrieved_laws_list = [rag_pipeline(q_for_rag)['related_laws'] for q_for_rag in processed_questions_for_rag] # Need to store questions/results or modify loop
        # retrieval_precision, retrieval_recall = calculate_retrieval_metrics(ground_truth, retrieved_laws_list) # Need to implement calculate_retrieval_metrics

    except Exception as e:
        print(f"An error occurred during potential metric calculation: {e}")
        print("Please ensure your predicted and ground truth answers are in a compatible format for the chosen metrics.")

## Load text encoder model

### Subtask:
Load the `keepitreal/vietnamese-sbert` model using the `sentence-transformers` library.

**Reasoning**:
The subtask is to load the text encoder model. This requires importing the necessary class and then loading the model.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the text encoder model
text_encoder_model = SentenceTransformer("keepitreal/vietnamese-sbert")

print("Successfully loaded the text encoder model: keepitreal/vietnamese-sbert")

Run the assessment - chạy đánh giá

In [ ]:
# The test data (test_task2) is already loaded in a previous cell (G4YAStN5xo8d).
# We will now call the main evaluation function (evaluate in IKMrDfWHzAtJ) with this data.

# Ensure the evaluate function is defined
if 'evaluate' in globals():
    print("Running assessment on test_task2 (multimodal):")
    # Call the evaluate function for the multimodal task
    # Note: The evaluation metrics will not be meaningful until the RAG pipeline
    # is fully implemented (retrieval and a suitable generation model) and the
    # evaluation logic is adapted to the test data structure.
    evaluate(test_task2, task_type='multimodal')
else:
    print("Error: evaluate function not found. Please run the cell that defines it (IKMrDfWHzAtJ).")

## Generate embeddings

### Subtask:
Generate vector embeddings for each law article in the `processed_law_df` DataFrame using the loaded Text Encoder model.

**Reasoning**:
Generate vector embeddings for each law article using the loaded text encoder model and store them as a NumPy array.

# Task
Update the RAG pipeline to use a suitable open-source, lightweight language model from Hugging Face for text generation, evaluate its performance on the provided test datasets ("test_task1" and "test_task2"), and ensure the pipeline is functional.

## Research potential models

### Subtask:
Identify open-source, lightweight language models on Hugging Face that are suitable for text generation or question answering in Vietnamese. Consider models based on different architectures (e.g., Causal LMs, Seq2Seq models).




### Subtask:
Choose a model from the research phase to attempt loading and using for generation.


**Reasoning**:
Based on the previous research, the `vinai/phobert-base` model was initially attempted but proved unsuitable for text generation in the current configuration. A more appropriate model designed for text generation is needed. Given the constraints of being open-source and lightweight, a smaller model with a causal language modeling architecture would be preferable. While a perfectly "lightweight" Vietnamese generation model might be scarce, I will attempt to use a smaller, general-purpose causal language model that has shown some capability in other languages and might perform reasonably with prompting in Vietnamese, or explicitly acknowledge the need for a specialized Vietnamese generation model if a suitable lightweight one isn't readily available. For this step, I will select `gpt2` as a widely available small CausalLM, understanding it's not Vietnamese-specific, and note that a fine-tuned or specifically Vietnamese generation model would be better in a real-world scenario. This allows the pipeline structure to be tested. I will load the tokenizer and model.



## Load text encoder model

### Subtask:
Load the `keepitreal/vietnamese-sbert` model using the `sentence-transformers` library.

**Reasoning**:
The subtask is to load the text encoder model. This requires importing the necessary class and then loading the model.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the text encoder model
text_encoder_model = SentenceTransformer("keepitreal/vietnamese-sbert")

print("Successfully loaded the text encoder model: keepitreal/vietnamese-sbert")

## Generate embeddings

### Subtask:
Generate vector embeddings for each law article in the `processed_law_df` DataFrame using the loaded Text Encoder model.

**Reasoning**:
Generate vector embeddings for each law article using the loaded text encoder model and store them as a NumPy array.

In [ ]:
import numpy as np

# Extract the 'law_text' column
law_texts = processed_law_df['law_text']

# Generate embeddings for the law texts
# The encode method handles batching automatically
law_embeddings = text_encoder_model.encode(law_texts.tolist(), show_progress_bar=True)

# Store the embeddings as a NumPy array (already is a NumPy array by default)
print(f"Generated embeddings with shape: {law_embeddings.shape}")

# Display the first few embeddings as an example
print("\nFirst 5 embeddings:")
print(law_embeddings[:5])

## Build faiss index

### Subtask:
Create a FAISS index from the generated embeddings stored in the `law_embeddings` NumPy array.

**Reasoning**:
Create a FAISS index from the generated embeddings.

In [ ]:
import faiss

# Get the dimension of the embeddings
embedding_dimension = law_embeddings.shape[1]

# Create a FAISS index (FlatL2 index for exact nearest neighbor search)
index = faiss.IndexFlatL2(embedding_dimension)

# Add the law embeddings to the index
index.add(law_embeddings)

# Print the number of vectors added to the index
print(f"Number of vectors added to the index: {index.ntotal}")

In [ ]:
!pip install faiss-cpu